# 10. Score Scaling and Cutoff Analysis

## Objective

This notebook provides a short business-oriented interpretation of the scorecard output.

It uses the predictions and reports generated in Notebook 07 and focuses on:

- score scaling parameters;
- score decile interpretation;
- PD cutoff scenarios;
- approval rate vs portfolio bad rate trade-off;
- recommended business interpretation.
 
This notebook uses the PD cutoff and score decile outputs already generated by the final logistic regression scorecard notebook.


In [ ]:
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

## 1. Paths and input files

In [ ]:
PROJECT_ROOT = Path("..")

INPUT_DIR = PROJECT_ROOT / "data" / "processed" / "07.logistic_regression_scorecard"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "10.score_scaling_and_cutoffs"
CHARTS_DIR = OUTPUT_DIR / "charts"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

REPORT_FILE = INPUT_DIR / "logistic_regression_scorecard_report.xlsx"
ARTIFACTS_FILE = INPUT_DIR / "logistic_regression_scorecard_artifacts.pkl"

if not REPORT_FILE.exists():
    raise FileNotFoundError(f"Missing expected Excel report from Notebook 07: {REPORT_FILE}")

print(f"Input report: {REPORT_FILE.resolve()}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")

## 2. Load scorecard outputs from Notebook 07

In [ ]:
# Core tables from Notebook 07
score_scaling = pd.read_excel(REPORT_FILE, sheet_name="score_scaling")
score_deciles = pd.read_excel(REPORT_FILE, sheet_name="score_deciles")
cutoff_review = pd.read_excel(REPORT_FILE, sheet_name="cutoff_review")
performance = pd.read_excel(REPORT_FILE, sheet_name="performance")

print("Loaded sheets:")
print("- score_scaling")
print("- score_deciles")
print("- cutoff_review")
print("- performance")

display(score_scaling)
display(score_deciles.head())
display(cutoff_review.head())

## 3. Score scaling interpretation

The scorecard was scaled using:

- Base Score = 600
- Base Odds = 20:1 good/bad
- PDO = 50

This means that an account with 20:1 good/bad odds receives a score of approximately 600.

Higher score means lower predicted default risk.


In [ ]:
score_scaling

## 4. Score decile analysis

The score decile report is the main business interpretation table.

Decile 1 contains the highest-score / lowest-risk borrowers.  
Decile 10 contains the lowest-score / highest-risk borrowers.

A good scorecard should show a clear increase in observed bad rate from high-score to low-score deciles.


In [ ]:
score_deciles_selected = score_deciles.copy()


wanted_cols = [
    "sample",
    "score_decile",
    "accounts",
    "bads",
    "observed_bad_rate",
    "avg_pd",
    "min_score",
    "avg_score",
    "max_score",
]

score_deciles_selected = score_deciles_selected[
    [c for c in wanted_cols if c in score_deciles_selected.columns]
]

score_deciles_selected

In [ ]:
for sample_name in ["train", "validation", "oot_test"]:
    data = score_deciles_selected[
        score_deciles_selected["sample"] == sample_name
    ].copy()

    if data.empty:
        continue

    fig, ax1 = plt.subplots(figsize=(10, 5))

    ax1.bar(
        data["score_decile"].astype(str),
        data["accounts"],
        alpha=0.75,
        label="Accounts",
    )
    ax1.set_xlabel("Score decile: 1 = best score / lowest risk")
    ax1.set_ylabel("Accounts")
    ax1.grid(axis="y", linestyle="--", alpha=0.3)

    ax2 = ax1.twinx()
    ax2.plot(
        data["score_decile"].astype(str),
        data["observed_bad_rate"],
        color="red",
        marker="o",
        linewidth=2.5,
        label="Observed bad rate",
    )

    if "avg_pd" in data.columns:
        ax2.plot(
            data["score_decile"].astype(str),
            data["avg_pd"],
            marker="x",
            linestyle="--",
            linewidth=2,
            label="Average predicted PD",
        )

    ax2.set_ylabel("Bad rate / PD", color="red")
    ax2.tick_params(axis="y", labelcolor="red")
    ax2.legend(loc="upper left")

    plt.title(f"Score Decile Risk Profile - {sample_name}")
    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"score_decile_risk_profile_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 5. PD cutoff scenarios

The cutoff logic is:

```text
Approve if predicted PD <= cutoff
```

Lower PD cutoffs are more conservative.  
Higher PD cutoffs are more growth-oriented.


In [ ]:
scenario_definitions = pd.DataFrame(
    [
        {
            "strategy": "Conservative",
            "pd_cutoff": 0.10,
            "interpretation": "Strict approval rule; lower approval rate and better expected portfolio quality.",
        },
        {
            "strategy": "Balanced",
            "pd_cutoff": 0.20,
            "interpretation": "Balanced approval and portfolio risk trade-off.",
        },
        {
            "strategy": "Growth",
            "pd_cutoff": 0.30,
            "interpretation": "Higher approval rate with higher accepted portfolio risk.",
        },
    ]
)

scenario_results = cutoff_review.merge(
    scenario_definitions,
    on="pd_cutoff",
    how="inner",
)

scenario_results = scenario_results[
    [
        "sample",
        "strategy",
        "pd_cutoff",
        "approval_rate",
        "rejection_rate",
        "approved_accounts",
        "rejected_accounts",
        "approved_bad_rate",
        "rejected_bad_rate",
        "bad_capture_in_rejected",
        "interpretation",
    ]
].sort_values(["sample", "pd_cutoff"])

scenario_results

In [ ]:
for sample_name in ["validation", "oot_test"]:
    data = scenario_results[scenario_results["sample"] == sample_name].copy()

    if data.empty:
        continue

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.bar(
        data["strategy"],
        data["approval_rate"],
        alpha=0.75,
        label="Approval rate",
    )
    ax.plot(
        data["strategy"],
        data["approved_bad_rate"],
        color="red",
        marker="o",
        linewidth=2.5,
        label="Approved bad rate",
    )

    ax.set_ylabel("Rate")
    ax.set_title(f"Approval Strategy Scenarios - {sample_name}")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.3)

    plt.tight_layout()
    plt.savefig(CHARTS_DIR / f"approval_strategy_scenarios_{sample_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 6. Reference strategy

For the purpose of this case study, the Balanced strategy is treated as the reference business scenario.

In [ ]:
recommended_strategy = "Balanced"

recommended_scenario = scenario_results[
    (scenario_results["strategy"] == recommended_strategy)
    & (scenario_results["sample"] == "oot_test")
].copy()

recommended_scenario

## 7. Save outputs

In [ ]:
outputs = {
    "score_scaling": score_scaling,
    "score_deciles": score_deciles_selected,
    "cutoff_review": cutoff_review,
    "scenario_definitions": scenario_definitions,
    "scenario_results": scenario_results,
    "recommended_strategy": recommended_strategy,
    "recommended_scenario": recommended_scenario,
}

with open(OUTPUT_DIR / "score_scaling_and_cutoffs_short_artifacts.pkl", "wb") as f:
    pickle.dump(outputs, f)

excel_path = OUTPUT_DIR / "score_scaling_and_cutoffs_short_report.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    score_scaling.to_excel(writer, sheet_name="score_scaling", index=False)
    score_deciles_selected.to_excel(writer, sheet_name="score_deciles", index=False)
    cutoff_review.to_excel(writer, sheet_name="cutoff_review", index=False)
    scenario_definitions.to_excel(writer, sheet_name="scenario_definitions", index=False)
    scenario_results.to_excel(writer, sheet_name="scenario_results", index=False)
    recommended_scenario.to_excel(writer, sheet_name="recommended_scenario", index=False)

print("Saved outputs:")
print(f"- {OUTPUT_DIR / 'score_scaling_and_cutoffs_short_artifacts.pkl'}")
print(f"- {excel_path}")
print(f"- charts saved in {CHARTS_DIR}")

## 8. Conclusions

The score scaling and cutoff analysis translates the statistical model output into an operational credit decisioning view.

The scorecard scaling uses:

- Base Score = 600
- Base Odds = 20:1
- PDO = 50

The score decile analysis confirms that the model ranks borrowers in the expected direction. Higher-score deciles have lower observed bad rates and lower average predicted PDs, while lower-score deciles concentrate materially higher credit risk.

The cutoff analysis is based on predicted PD thresholds rather than arbitrary score thresholds. This is easier to interpret from a business perspective:

- lower PD cutoffs are more conservative;
- higher PD cutoffs support higher approval volumes but accept more risk;
- rejected populations have materially higher observed bad rates, confirming that the model separates risk effectively.

Three illustrative strategies were reviewed:

- Conservative: PD cutoff of 10%;
- Balanced: PD cutoff of 20%;
- Growth: PD cutoff of 30%.

For this case study, the Balanced scenario is used as the reference strategy because it provides a reasonable middle ground between approval volume and portfolio risk.

The selected cutoff should not be treated as a final policy recommendation, since iother factors should be considered in a real implementation, including:
- expected profitability;
- pricing;
- LGD, EAD;
- operational capacity, approval targets;
- defined risk appetite and regulatory considerations.